In [4]:
from pathlib import Path
import ast

import pandas as pd

In [ ]:
EXTRACTED_ENTITIES_FP = "PATH/TO/FILE.CSV"

In [ ]:
fp = Path(EXTRACTED_ENTITIES_FP)

# fp = Path("data/extracted_entities_2026_06_03.csv")
df_input = pd.read_csv(fp)
df_input = df_input.rename(columns={"geographic location" : "location"})

#, index_col=0)
# Entity values to python objects
entity_cols = ['person', 'location', 'organization']
df_input[entity_cols] = df_input[entity_cols].map(ast.literal_eval)
df_input

,filename,paragraph,sentence,person,location,organization,historical_event
0,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","Bij het bezoek, dat door den Koning dezer dage...","[{'start': 89, 'end': 94, 'text': 'Z. M.', 'la...",[],[],[]
1,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","adjudant den majoor A. E. Mansveldt, Zr.","[{'start': 20, 'end': 35, 'text': 'A. E. Mansv...",[],[],[]
2,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...",Ms. stalmeester W. C. baron Snouckaert van Sch...,[],[],[],[]
3,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","ordonnansofiicier, den kapitein H. J. M. C. ba...",[],[],[],[]
4,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...",H. M. de Koningin zal vergezeld worden door me...,"[{'start': 0, 'end': 17, 'text': 'H. M. de Kon...",[],[],[]
...,...,...,...,...,...,...,...
46460,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"â Te .Muntok en Herawang viek vele regens, i...",[],"[{'start': 8, 'end': 14, 'text': 'Muntok', 'la...",[],"[{'start': 78, 'end': 83, 'text': 'wedei', 'la..."
46461,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,n de 52 op nltinie !â¢ â¢ â¢ ml.er handelin...,[],[],[],[]
46462,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"i overhed I persoon, terwijl in het distrikt P...",[],[],[],"[{'start': 65, 'end': 76, 'text': 'Januarij 11..."
46463,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"Dt oogst, waarmede zich de bevolking onledig h...",[],[],[],[]


In [6]:
# Optional: reduce irrelevant names by only keeping sentences that contain 'Nias'
# and the sentences before and after it (if they are from the same article)

df = df_input.copy()

# If you don't want to filter, comment out the following lines:

df['relevant_sentence'] = False
df['relevant_sentence'] = df.eval("sentence.str.contains('Nias')")
# Also assign sentence before and after to relevant_sentence (if it's from the same article (i.e. filename))
df['is_same_as_previous'] = df['filename'] == df['filename'].shift(1)
df['is_same_as_next'] = df['filename'] == df['filename'].shift(-1)

# Set relevant_sentence to True for sentences before and after relevant sentences
df['relevant_sentence'] = df['relevant_sentence'] | (df['is_same_as_previous'] & df['relevant_sentence'].shift(1, fill_value=False)) | (df['is_same_as_next'] & df['relevant_sentence'].shift(-1, fill_value=False))


df = df.query("relevant_sentence")
df

,filename,paragraph,sentence,person,location,organization,historical_event,relevant_sentence,is_same_as_previous,is_same_as_next
30,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...",Kaakebeen en 25 onder-officieren en manschappe...,[],"[{'start': 55, 'end': 70, 'text': 'Goenong Sit...",[],[],True,True,True
31,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","civilen en militairen gezaghebber van Nias, de...",[],"[{'start': 38, 'end': 42, 'text': 'Nias', 'lab...",[],[],True,True,True
32,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","Meijer, 19 soldaten qn een mortier","[{'start': 0, 'end': 6, 'text': 'Meijer', 'lab...",[],[],[],True,True,True
149,MMKB19_000142109_mpeg21_a00028.json,"uv ci ^ BATAVIA, den 29â Maart 1863. Het alg...",Deze aardbeving was groot genoeg om van zich t...,[],"[{'start': 92, 'end': 98, 'text': 'Onrust', 'l...",[],"[{'start': 0, 'end': 15, 'text': 'Deze aardbev...",True,True,True
150,MMKB19_000142109_mpeg21_a00028.json,"uv ci ^ BATAVIA, den 29â Maart 1863. Het alg...",//Veel onaangenamer gebeurtenissen hebben er e...,[],"[{'start': 55, 'end': 62, 'text': 'Sumatra', '...",[],[],True,True,True
...,...,...,...,...,...,...,...,...,...,...
46445,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"Itel /.. K. stc Maat â 'i Wind, die in de eer...",[],"[{'start': 72, 'end': 78, 'text': 'Padang', 'l...",[],[],True,True,True
46446,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,tevens de kontroleur van laatstgenoemd eiland ...,[],[],[],[],True,True,True
46452,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"De vele regens waren, vooral in bel Sol (Padan...",[],"[{'start': 32, 'end': 39, 'text': 'bel Sol', '...",[],"[{'start': 0, 'end': 14, 'text': 'De vele rege...",True,True,True
46453,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"Aan de piangans, welke in het N padigewas scha...",[],"[{'start': 85, 'end': 89, 'text': 'Nias', 'lab...",[],[],True,True,True


### Create dataframe with all persons

In [8]:
# Create a dataframe with only persons, one row per person
persons = df[['filename', 'person']]\
    .explode('person')\
    .dropna(subset=['person'])\
    .reset_index(drop=True)

persons_df = pd.concat([persons, pd.DataFrame(persons['person'].tolist())], axis=1).drop(columns=['person'])
print(f"Unieke personen: {persons_df.text.nunique()}")
persons_df

Unieke personen: 320


,filename,start,end,text,label,score
0,MMKB19_000142099_mpeg21_a00003.json,0,6,Meijer,persoon (naam),0.877176
1,MMKB19_000142109_mpeg21_a00028.json,112,128,Reinier Claeszen,persoon (naam),0.783604
2,MMKB19_000142109_mpeg21_a00028.json,346,355,Maleijers,persoon (naam),0.625696
3,MMKB19_000142123_mpeg21_a00020.json,96,108,heer Francis,persoon (naam),0.641545
4,MMKB19_000142123_mpeg21_a00020.json,188,204,heer van Kerchem,persoon (naam),0.743865
...,...,...,...,...,...,...
686,ddd_110532335_mpeg21_a0040.json,3,8,Julij,persoonsnaam,0.784169
687,ddd_110532841_mpeg21_a0043.json,0,11,Willemsorde,persoonsnaam,0.607359
688,ddd_110532849_mpeg21_a0033.json,6,22,W. S. J. Docters,persoonsnaam,0.611494
689,ddd_110533902_mpeg21_a0021.json,67,75,de Sahan,persoonsnaam,0.679414


## Convert to embeddings, build and search index

We're creating a 'FAISS' vector-index, to store the embeddings, and which we can use to quickly search and retrieve candidates (https://github.com/facebookresearch/faiss). Below is just a basic index, we could optimize it a bit if it turns out to be too slow

In [10]:
from sentence_transformers import SentenceTransformer
import faiss

# https://rapidfuzz.github.io/RapidFuzz/index.html
from rapidfuzz import fuzz, distance

In [ ]:
# embed_model = "emanjavacas/GysBERT"
embed_model = "intfloat/multilingual-e5-small"

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer(embed_model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7456.91it/s]


In [53]:
## Create embeddings for all names in the dataframe
# Get list of unique names
names = persons_df.text.unique().tolist()
print(f"Number of unique names: {len(names)}")

# Create embeddings for the unique names
embeddings = model.encode(names)

# Normalize embeddings
faiss.normalize_L2(embeddings)


# Create index
index = faiss.IndexFlatL2(embeddings.shape[1])
print(index.is_trained)

# Add embeddings to the index
index.add(embeddings)

print(index.ntotal)

Number of unique names: 320
True
320


In [ ]:
# Find k nearest neighbors for each name
k = 16

CANDIDATE_DISTANCES, CANDIDATE_INDICES = index.search(embeddings, k)

### Inspect results

In [31]:
def add_distance_metrics(df, name_col='name', candidate_col='candidate', name_idx_col='name_id', candidate_idx_col='candidate_id'):
    """
    Add distance metrics to the DataFrame for each name-candidate pair.
    
    Parameters:
    - df: DataFrame containing name-candidate pairs
    - name_col: Column name for the original name
    - candidate_col: Column name for the candidate name
    - name_idx_col: Column name for the index of the original name
    - candidate_idx_col: Column name for the index of the candidate name
    
    Returns:
    - DataFrame with added distance metrics
    """
    
    # Levenshtein weights for insertion, deletion, substitution
    levenshtein_weights = (2, 2, 1)  
    
    # Add columns with fuzzy matching scores
    df['wratio'] = df.apply(lambda x: fuzz.WRatio(df[name_col].iloc[0], x[candidate_col]), axis=1)
    df['token_sort_ratio'] = df.apply(lambda x: fuzz.token_sort_ratio(df[name_col].iloc[0], x[candidate_col]), axis=1)
    df['lcs_ratio'] = df.apply(lambda x: distance.LCSseq.normalized_distance(df[name_col].iloc[0].lower(), x[candidate_col].lower()), axis=1)
    df['levenshtein_ratio'] = df.apply(lambda x: distance.Levenshtein.normalized_distance(df[name_col].iloc[0].lower(), x[candidate_col].lower(), weights=levenshtein_weights), axis=1)
    df['jarowinkler'] = df.apply(lambda x: distance.JaroWinkler.normalized_distance(df[name_col].iloc[0].lower(), x[candidate_col].lower()), axis=1)
    
    return df

In [32]:
# Create single dataframe with all results for all names, instead of just one name
names_series = pd.Series(names)

# Flatten the candidate indices and distances for all names
candidate_indices_flat = CANDIDATE_INDICES.flatten()
candidate_distances_flat = CANDIDATE_DISTANCES.flatten()
# Create a DataFrame with all results for all names
all_results = pd.DataFrame({
    "name": names_series.repeat(k).values,
    "name_id": names_series.index.repeat(k).values,
    "candidate": [names[j] for j in candidate_indices_flat],
    "candidate_id": candidate_indices_flat,
    "l2_distance": candidate_distances_flat
})

# Drop the first row for each name (the name itself) from the results (i.e., where name_id == candidate_id)
all_results = all_results.query("name_id != candidate_id")\
    .reset_index(drop=True)

all_results = add_distance_metrics(all_results)

all_results

,name,name_id,candidate,candidate_id,l2_distance,wratio,token_sort_ratio,lcs_ratio,levenshtein_ratio,jarowinkler
0,Meijer,0,luitenant Meijer,151,0.099623,90.000000,54.545455,0.625000,0.769231,0.548611
1,Meijer,0,Luitenant Meijer,239,0.107859,90.000000,54.545455,0.625000,0.769231,0.548611
2,Meijer,0,Maleijer,256,0.133420,85.714286,85.714286,0.250000,0.400000,0.075000
3,Meijer,0,F. W. Meijer,281,0.165139,90.000000,66.666667,0.500000,0.666667,0.583333
4,Meijer,0,De maleijer,45,0.179313,81.818182,58.823529,0.454545,0.625000,0.520202
...,...,...,...,...,...,...,...,...,...,...
4795,de Sahan,319,Een der deensche officieren,287,0.278093,49.090909,24.242424,0.851852,0.916667,0.462963
4796,de Sahan,319,Si IsmaÃ,126,0.280914,14.285714,14.285714,0.875000,0.900000,0.569444
4797,de Sahan,319,den Niasser,176,0.281848,47.058824,35.294118,0.636364,0.750000,0.494949
4798,de Sahan,319,De pogiog,296,0.282120,30.000000,26.666667,0.777778,0.916667,0.574074


In [34]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

In [35]:
filename = f"data/name_candidate_distances_{timestamp}.csv"

print(f"Saving results to {filename}")
all_results.to_csv(filename, index=False)

Saving results to data/name_candidate_distances_2026_07_23_14_48_30.csv


In [52]:
block_id = 3
all_results.query(f"name_id == {block_id}")

,name,name_id,candidate,candidate_id,l2_distance,wratio,token_sort_ratio,lcs_ratio,levenshtein_ratio,jarowinkler
45,heer Francis,3,heer Erancis,36,0.187544,57.000000,33.333333,0.750000,0.833333,0.416667
46,heer Francis,3,heer Sievers,271,0.194373,57.000000,33.333333,0.666667,0.777778,0.416667
47,heer Francis,3,heer van Kerchem,4,0.196777,54.000000,27.272727,0.812500,0.884615,0.437500
48,heer Francis,3,de heer J.,42,0.212094,57.000000,37.500000,0.700000,0.785714,0.400000
49,heer Francis,3,heer Steinmetz,14,0.212577,57.000000,40.000000,0.785714,0.863636,0.432540
50,heer Francis,3,de heer Stieltjes,56,0.217111,57.000000,34.782609,0.764706,0.857143,0.441176
51,heer Francis,3,De heer Kraijenhoff,247,0.218518,57.000000,40.000000,0.789474,0.875000,0.447368
52,heer Francis,3,De heer Kraijenhoft,145,0.221024,57.000000,40.000000,0.789474,0.875000,0.447368
53,heer Francis,3,heer ArriÃ«ns,15,0.247829,57.000000,31.578947,0.769231,0.900000,0.423077
54,heer Francis,3,majoor Fritzen,17,0.251105,34.200000,30.000000,0.785714,0.863636,0.428571
